# LongCat-Video-Avatar 1.5 (G4)

Audio-driven single-person video, BF16 + Whisper + 8-step DMD; outputs on Drive.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "longcat-avatar")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "7860"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- LongCat-Video-Avatar 1.5 on local SSD, Blackwell-compatible CUDA environment
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

LC_WORK = Path("/content/longcat-avatar")
LC_REPO = LC_WORK / "repo"
LC_MODEL = LC_WORK / "model"
LC_VENV = LC_WORK / "venv"
LC_REVISION = "6b3f4b8582a8bc3f20f795735f5383716c4ba794"
LC_WORK.mkdir(parents=True, exist_ok=True)
LC_OUTPUT = Path(DATA_DIR) / "output"
LC_OUTPUT.mkdir(parents=True, exist_ok=True)
LC_ENV = {**os.environ, "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0",
          "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}
# Compiler caches: one tar archive per environment/GPU on Drive, unpacked to local SSD.
(LC_WORK / "colab_cache.py").write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
sys.path.insert(0, str(LC_WORK))
from colab_cache import restore_cache, save_cache
LC_CACHE_KEY = "torch211-cu130-" + re.sub(r"[^A-Za-z0-9_.-]+", "_", ct_gpu_stat("name").splitlines()[0])
LC_CACHE_ARCHIVES = Path(DATA_DIR) / "kernel-archives"
LC_CACHE_PAIRS = []
for name in ("triton-cache", "inductor-cache"):
    local_cache = LC_WORK / name
    try:
        restore_cache(local_cache, Path(DATA_DIR) / name, LC_CACHE_ARCHIVES, LC_CACHE_KEY)
    except (OSError, ValueError) as err:
        print(f"Cache restore warning ({name}): {err}", flush=True)
    LC_CACHE_PAIRS.append(local_cache)

if not (LC_REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/meituan-longcat/LongCat-Video.git", str(LC_REPO)], check=True)
subprocess.run(["git", "-C", str(LC_REPO), "checkout", "--detach", LC_REVISION], check=True)

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if not (LC_VENV / "bin/python").exists():
    subprocess.run([UV, "venv", "--python", "3.11", "--seed", str(LC_VENV)], check=True)
LC_PYTHON = str(LC_VENV / "bin/python")

def lc_install():
    marker = LC_WORK / "environment-v1.ok"
    if marker.exists():
        return
    log_path = LC_WORK / "install.log"
    with log_path.open("a") as log_file:
        def run(args):
            subprocess.run(args, stdout=log_file, stderr=subprocess.STDOUT, env=LC_ENV, check=True)
        # Upstream's torch 2.6/cu124 predates sm_120; use CUDA 13 wheels for G4.
        run([UV, "pip", "install", "--python", LC_PYTHON, "torch==2.11.0", "torchvision==0.26.0", "torchaudio==2.11.0",
             "--index-url", "https://download.pytorch.org/whl/cu130"])
        run([UV, "pip", "install", "--python", LC_PYTHON,
             "numpy==1.26.4", "transformers==4.41.0", "diffusers==0.35.1",
             "einops==0.8.0", "ftfy==6.2.0", "loguru==0.7.2", "psutil",
             "av==12.0.0", "opencv-python-headless==4.9.0.80", "imageio==2.37.0",
             "imageio-ffmpeg==0.6.0", "huggingface-hub==0.36.0", "hf_xet",
             "accelerate>=1,<2", "sentencepiece", "librosa==0.11.0", "pyloudnorm==0.1.1", "soundfile==0.13.1", "scipy==1.15.3", "gradio==5.49.1", "uvicorn", "fastapi"])
    marker.write_text("ok\n")

def lc_download():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface-hub", "hf_xet"], check=True)
    download = r"""
from huggingface_hub import snapshot_download
import sys
from pathlib import Path
root = Path(sys.argv[1])
snapshot_download('meituan-longcat/LongCat-Video', local_dir=root / 'LongCat-Video',
                  allow_patterns=['tokenizer/*', 'text_encoder/*', 'vae/*'])
snapshot_download('meituan-longcat/LongCat-Video-Avatar-1.5', local_dir=root / 'LongCat-Video-Avatar-1.5',
                  allow_patterns=['base_model/*', 'scheduler/*', 'lora/dmd_lora.safetensors',
                                  'whisper-large-v3/*.json', 'whisper-large-v3/*.txt',
                                  'whisper-large-v3/model.safetensors'])
"""
    with (LC_WORK / "download.log").open("a") as log_file:
        subprocess.run([sys.executable, "-c", download, str(LC_MODEL)], stdout=log_file, stderr=subprocess.STDOUT,
                       env=LC_ENV, check=True)
    shutil.rmtree("/root/.cache/huggingface/xet", ignore_errors=True)

with ThreadPoolExecutor(max_workers=2) as pool:
    jobs = [pool.submit(lc_install), pool.submit(lc_download)]
    while not all(job.done() for job in jobs):
        for job in jobs:
            if job.done():
                job.result()
        print("LongCat setup: environment=" + ("ready" if jobs[0].done() else "installing") +
              " model=" + ("ready" if jobs[1].done() else "downloading") +
              f" SSD free={shutil.disk_usage('/content').free / 1e9:.1f} GB; logs={LC_WORK}", flush=True)
        time.sleep(20)
    for job in jobs:
        job.result()
print("LongCat environment and model ready", flush=True)


In [ ]:
# @title 2. Service -- load Avatar 1.5 BF16 + Whisper + DMD, verify inference, then serve Gradio on loopback
(LC_WORK / 'longcat_video_app.py').write_text('"""Single-GPU LongCat-Video UI; embedded into the self-contained Colab notebook."""\nimport json\nimport os\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.nn.attention import SDPBackend, sdpa_kernel\n\n\ndef install_sdpa():\n    """Use native fused SDPA instead of the upstream CUDA-12.4 FlashAttention build.\n\n    Cross attention uses packed, variable-length text; split each sample before\n    calling SDPA so padding and neighbouring prompts never enter its attention.\n    This adapter is intentionally single GPU, matching the Colab deployment.\n    """\n    from longcat_video.modules.attention import Attention, MultiHeadCrossAttention\n\n    def self_attention(self, q, k, v, shape):\n        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):\n            return F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, scale=self.scale)\n\n    def cross_attention(self, x, cond, kv_seqlen):\n        batch, tokens, channels = x.shape\n        q = self.q_norm(self.q_linear(x).view(1, -1, self.num_heads, self.head_dim))\n        kv = self.kv_linear(cond).view(1, -1, 2, self.num_heads, self.head_dim)\n        k, v = kv.unbind(2)\n        k = self.k_norm(k)\n        outputs, offset = [], 0\n        for i, length in enumerate(kv_seqlen):\n            qi = q[:, i * tokens:(i + 1) * tokens].transpose(1, 2)\n            ki = k[:, offset:offset + length].transpose(1, 2)\n            vi = v[:, offset:offset + length].transpose(1, 2)\n            with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):\n                out = F.scaled_dot_product_attention(qi, ki, vi, dropout_p=0.0)\n            outputs.append(out.transpose(1, 2))\n            offset += length\n        return self.proj(torch.cat(outputs, dim=1).reshape(batch, tokens, channels))\n\n    Attention._process_attn = self_attention\n    MultiHeadCrossAttention._process_cross_attn = cross_attention\n\n\ndef load_pipeline():\n    from transformers import AutoTokenizer, UMT5EncoderModel\n    from longcat_video.pipeline_longcat_video import LongCatVideoPipeline\n    from longcat_video.modules.scheduling_flow_match_euler_discrete import FlowMatchEulerDiscreteScheduler\n    from longcat_video.modules.autoencoder_kl_wan import AutoencoderKLWan\n    from longcat_video.modules.longcat_video_dit import LongCatVideoTransformer3DModel\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("LongCat-Video requires a CUDA GPU")\n    install_sdpa()\n    model = os.environ["CT_LONGCAT_MODEL_DIR"]\n    common = dict(torch_dtype=torch.bfloat16, local_files_only=True)\n    tokenizer = AutoTokenizer.from_pretrained(model, subfolder="tokenizer", local_files_only=True)\n    text_encoder = UMT5EncoderModel.from_pretrained(model, subfolder="text_encoder", **common)\n    vae = AutoencoderKLWan.from_pretrained(model, subfolder="vae", **common)\n    dit = LongCatVideoTransformer3DModel.from_pretrained(\n        model, subfolder="dit", cp_split_hw=[1, 1],\n        enable_flashattn2=False, enable_flashattn3=False, enable_xformers=False,\n        **common,\n    )\n    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(model, subfolder="scheduler")\n    for module in (text_encoder, vae, dit):\n        module.eval().requires_grad_(False)\n    pipe = LongCatVideoPipeline(tokenizer, text_encoder, vae, scheduler, dit)\n    pipe.dit.load_lora(str(Path(model) / "lora/cfg_step_lora.safetensors"), "cfg_step_lora")\n    pipe.to("cuda")\n    return pipe\n\n\ndef main():\n    import gradio as gr\n    import uvicorn\n    from fastapi import FastAPI\n    from diffusers.utils import export_to_video, load_image, load_video\n\n    output_dir = Path(os.environ["CT_LONGCAT_OUTPUT_DIR"])\n    output_dir.mkdir(parents=True, exist_ok=True)\n    pipe = load_pipeline()\n    lock = threading.Lock()\n    # A real inference pass is the readiness gate, not just an HTTP listener.\n    with torch.inference_mode():\n        smoke = pipe.generate_t2v(\n            prompt="A small orange cat walking through a sunny garden.",\n            height=256, width=256, num_frames=9, num_inference_steps=1,\n            guidance_scale=1.0, generator=torch.Generator(device="cuda").manual_seed(42),\n        )[0]\n    smoke_path = Path(os.environ["CT_LONGCAT_WORK_DIR"]) / "smoke.mp4"\n    export_to_video(smoke, str(smoke_path), fps=15)\n    print(f"LONGCAT_INFERENCE_OK frames={len(smoke)} output={smoke_path}", flush=True)\n    del smoke\n    torch.cuda.empty_cache()\n\n    def generate(task, prompt, negative, image, video, resolution, width, height,\n                 frames, steps, guidance, distill, seed, segments):\n        if not prompt.strip():\n            raise gr.Error("Enter a video description")\n        if task == "Image to Video" and not image:\n            raise gr.Error("Upload a reference image")\n        if task == "Video Continuation" and not video:\n            raise gr.Error("Upload a reference video")\n        width, height, frames = int(width), int(height), int(frames)\n        if not (256 <= width <= 1024 and 256 <= height <= 1024 and 17 <= frames <= 161):\n            raise gr.Error("Width/height must be 256-1024 and frames 17-161")\n        if width % 16 or height % 16 or frames % 4 != 1:\n            raise gr.Error("Width/height must be multiples of 16 and frames must be 4n+1")\n        if task not in ("Text to Video", "Image to Video", "Video Continuation") or resolution not in ("480p", "720p"):\n            raise gr.Error("Choose a valid generation mode and resolution")\n        if not (1 <= int(segments) <= 10 and 1 <= int(steps) <= 100 and 1 <= float(guidance) <= 10):\n            raise gr.Error("Segments, steps or guidance scale out of range")\n        seed = int(seed)\n        if seed < 0:\n            seed = int.from_bytes(os.urandom(4), "little")\n        started = time.time()\n        with lock, torch.inference_mode():\n            try:\n                if distill:\n                    pipe.dit.enable_loras(["cfg_step_lora"])\n                generator = torch.Generator(device="cuda").manual_seed(seed)\n                kwargs = dict(prompt=prompt, negative_prompt=None if distill else negative,\n                              num_frames=frames, num_inference_steps=16 if distill else int(steps),\n                              guidance_scale=1.0 if distill else float(guidance),\n                              use_distill=distill, generator=generator)\n                if task == "Text to Video":\n                    if int(segments) > 1:\n                        from PIL import Image\n                        # Continuation uses upstream aspect buckets: select the same\n                        # bucket for the first segment so every clip can be joined.\n                        height, width = pipe.get_condition_shape(\n                            Image.new("RGB", (width, height)), resolution,\n                            scale_factor_spatial=pipe.vae_scale_factor_spatial * 2,\n                        )\n                    result = pipe.generate_t2v(height=height, width=width, **kwargs)[0]\n                elif task == "Image to Video":\n                    result = pipe.generate_i2v(image=load_image(image), resolution=resolution, **kwargs)[0]\n                else:\n                    import cv2\n                    cap = cv2.VideoCapture(video)\n                    fps = cap.get(cv2.CAP_PROP_FPS) or 15\n                    cap.release()\n                    source = load_video(video)[::max(1, round(fps / 15))]\n                    if len(source) < 13:\n                        raise gr.Error("The reference video needs at least 13 frames (sampled at 15 fps)")\n                    result = pipe.generate_vc(video=source, resolution=resolution, num_cond_frames=13,\n                                              use_kv_cache=True, offload_kv_cache=True,\n                                              enhance_hf=not distill, **kwargs)[0]\n                clips = [result]\n                for _ in range(1, int(segments)):\n                    from PIL import Image\n                    source = [Image.fromarray((frame * 255).clip(0, 255).astype(np.uint8)) for frame in result[-13:]]\n                    result = pipe.generate_vc(video=source, resolution=resolution, num_cond_frames=13,\n                                              use_kv_cache=True, offload_kv_cache=True,\n                                              enhance_hf=not distill, **kwargs)[0]\n                    clips.append(result[13:])\n                result = np.concatenate(clips, axis=0)\n                name = f"{time.strftime(\'%Y%m%d-%H%M%S\')}-{uuid.uuid4().hex[:8]}"\n                path = output_dir / f"{name}.mp4"\n                export_to_video(result, str(path), fps=15)\n                metadata = dict(task=task, prompt=prompt, negative_prompt=negative, seed=seed,\n                                frames=len(result), fps=15, resolution=resolution,\n                                width=int(result.shape[2]), height=int(result.shape[1]),\n                                steps=16 if distill else int(steps), guidance=1.0 if distill else guidance,\n                                distill=distill, segments=int(segments), elapsed_seconds=round(time.time()-started, 1))\n                path.with_suffix(\'.json\').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))\n                return str(path), str(path), metadata\n            finally:\n                pipe.dit.disable_all_loras()\n                torch.cuda.empty_cache()\n\n    with gr.Blocks(title="LongCat-Video") as demo:\n        gr.Markdown("# LongCat-Video\\nText to video · image to video · video continuation. Results are saved to Google Drive automatically.")\n        with gr.Row():\n            with gr.Column():\n                task = gr.Dropdown(["Text to Video", "Image to Video", "Video Continuation"], value="Text to Video", label="Generation mode")\n                prompt = gr.Textbox(label="Video description", lines=4, placeholder="Describe the scene, character actions and camera movement (English recommended)")\n                negative = gr.Textbox(label="Negative prompt", value="blurry, low quality, static, subtitles")\n                image = gr.Image(type="filepath", label="Reference image (image to video)")\n                video = gr.Video(label="Reference video (video continuation)")\n                distill = gr.Checkbox(value=True, label="Fast mode (16 steps)")\n                with gr.Accordion("Generation parameters", open=False):\n                    resolution = gr.Dropdown(["480p", "720p"], value="480p", label="Resolution for image / continuation / multi-segment generation")\n                    width = gr.Slider(256, 1024, value=832, step=16, label="Text-to-video width")\n                    height = gr.Slider(256, 1024, value=480, step=16, label="Text-to-video height")\n                    frames = gr.Slider(17, 161, value=93, step=4, label="Frames per segment (15 fps)")\n                    steps = gr.Slider(1, 100, value=50, step=1, label="Sampling steps (standard mode)")\n                    guidance = gr.Slider(1, 10, value=4, step=0.5, label="Guidance scale (standard mode)")\n                    seed = gr.Number(value=42, precision=0, label="Random seed (-1 for random)")\n                    segments = gr.Slider(1, 10, value=1, step=1, label="Number of consecutive segments")\n                submit = gr.Button("Generate video", variant="primary")\n            with gr.Column():\n                result = gr.Video(label="Result")\n                download = gr.File(label="Download video")\n                metadata = gr.JSON(label="Generation info")\n        submit.click(generate, [task, prompt, negative, image, video, resolution, width, height,\n                               frames, steps, guidance, distill, seed, segments],\n                     [result, download, metadata], api_name="generate", concurrency_limit=1)\n    api = FastAPI()\n\n    @api.get(\'/health\')\n    def health():\n        return {"status": "ok", "model": "LongCat-Video", "gpu": torch.cuda.get_device_name(0),\n                "inference_verified": True}\n\n    demo.queue(max_size=8, default_concurrency_limit=1)\n    public_host = os.environ.get("CT_PUBLIC_HOSTNAME", "")\n    api = gr.mount_gradio_app(\n        api, demo, path="/", allowed_paths=[str(output_dir)],\n        root_path=f"https://{public_host}" if public_host else None,\n    )\n    uvicorn.run(api, host="127.0.0.1", port=int(os.environ.get("CT_SERVICE_PORT", "7860")))\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")
(LC_WORK / 'longcat_avatar_app.py').write_text('"""Single-person LongCat-Video-Avatar 1.5 with authenticated Gradio hosting."""\nimport inspect\nimport json\nimport math\nimport os\nimport subprocess\nimport textwrap\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.nn.attention import SDPBackend, sdpa_kernel\n\nfrom longcat_video_app import install_sdpa\nfrom avatar_sampling import sampling_settings, apply_sampling_mode, plan_segments\n\n\ndef flash_attn_func(q, k, v, dropout_p=0.0, softmax_scale=None):\n    """Upstream BSHD convention, native fused SDPA on Blackwell."""\n    with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):\n        out = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2),\n                                            v.transpose(1, 2), dropout_p=dropout_p,\n                                            scale=softmax_scale)\n    return out.transpose(1, 2)\n\n\ndef install_avatar_sdpa():\n    install_sdpa()\n    from longcat_video.modules.attention import Attention as BaseAttention\n    from longcat_video.modules.avatar import attention as avatar\n    avatar.Attention._process_attn = BaseAttention._process_attn\n    # Keep the upstream audio projection, frame grouping, RoPE, and output\n    # projection intact; replace only its external FlashAttention call.\n    source = textwrap.dedent(inspect.getsource(avatar.SingleStreamAttention._process_cross_attn))\n    old = \'        from flash_attn import flash_attn_func\\n\'\n    if source.count(old) != 1:\n        raise RuntimeError(\'Pinned Avatar attention source changed; SDPA adapter needs review\')\n    source = source.replace(old, \'        flash_attn_func = _avatar_sdpa\\n\')\n    namespace = {**vars(avatar), \'_avatar_sdpa\': flash_attn_func}\n    exec(compile(source, \'<avatar-audio-sdpa>\', \'exec\'), namespace)\n    avatar.SingleStreamAttention._process_cross_attn = namespace[\'_process_cross_attn\']\n\n\ndef load_pipeline():\n    from transformers import AutoTokenizer, UMT5EncoderModel, WhisperModel, AutoFeatureExtractor\n    from longcat_video.pipeline_longcat_video_avatar import LongCatVideoAvatarPipeline\n    from longcat_video.modules.scheduling_flow_match_euler_discrete import FlowMatchEulerDiscreteScheduler\n    from longcat_video.modules.autoencoder_kl_wan import AutoencoderKLWan\n    from longcat_video.modules.avatar.longcat_video_dit_avatar import LongCatVideoAvatarTransformer3DModel\n\n    install_avatar_sdpa()\n    root = Path(os.environ[\'CT_LONGCAT_MODEL_DIR\'])\n    base, model = root / \'LongCat-Video\', root / \'LongCat-Video-Avatar-1.5\'\n    common = dict(torch_dtype=torch.bfloat16, local_files_only=True)\n    tokenizer = AutoTokenizer.from_pretrained(base, subfolder=\'tokenizer\', local_files_only=True)\n    text_encoder = UMT5EncoderModel.from_pretrained(base, subfolder=\'text_encoder\', **common)\n    vae = AutoencoderKLWan.from_pretrained(base, subfolder=\'vae\', **common)\n    dit = LongCatVideoAvatarTransformer3DModel.from_pretrained(\n        model, subfolder=\'base_model\', cp_split_hw=[1, 1],\n        enable_flashattn2=True, enable_flashattn3=False, enable_xformers=False, **common)\n    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(model, subfolder=\'scheduler\')\n    audio_encoder = WhisperModel.from_pretrained(model / \'whisper-large-v3\', local_files_only=True).cuda().eval()\n    extractor = AutoFeatureExtractor.from_pretrained(model / \'whisper-large-v3\', local_files_only=True)\n    for module in (text_encoder, vae, dit, audio_encoder):\n        module.eval().requires_grad_(False)\n    pipe = LongCatVideoAvatarPipeline(tokenizer, text_encoder, vae, scheduler, dit,\n                                      audio_encoder, extractor, model_type=\'avatar-v1.5\')\n    dit.load_lora(str(model / \'lora/dmd_lora.safetensors\'), \'dmd\')\n    pipe.to(\'cuda\')\n    dit.enable_loras([\'dmd\'])\n    return pipe\n\n\ndef window_audio(embeddings, start, frames):\n    offsets = torch.arange(5, device=embeddings.device) - 2\n    indices = torch.arange(start, start + frames, device=embeddings.device)[:, None] + offsets[None]\n    return embeddings[indices.clamp(0, len(embeddings)-1)][None].cuda()\n\n\ndef main():\n    import gradio as gr\n    import imageio_ffmpeg\n    import soundfile as sf\n    import uvicorn\n    from PIL import Image\n    from fastapi import FastAPI, HTTPException\n    from fastapi.responses import FileResponse\n    from longcat_avatar_drive import DRIVE_ROOT, resolve_drive, resolve_input, list_drive\n    from diffusers.utils import export_to_video, load_image\n\n    work = Path(os.environ[\'CT_LONGCAT_WORK_DIR\'])\n    output_dir = Path(os.environ[\'CT_LONGCAT_OUTPUT_DIR\'])\n    output_dir.mkdir(parents=True, exist_ok=True)\n    repo = Path(os.environ[\'CT_LONGCAT_REPO_DIR\'])\n    sample_audio, sample_image = repo / \'assets/avatar/single/man.mp3\', repo / \'assets/avatar/single/man.png\'\n    sample_prompt = json.loads((repo / \'assets/avatar/single_example_1.json\').read_text())[\'prompt\']\n    pipe = load_pipeline()\n    lock = threading.Lock()\n    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()\n\n    def prepare_audio(audio, seconds, target):\n        subprocess.run([ffmpeg, \'-y\', \'-loglevel\', \'error\', \'-i\', str(audio), \'-vn\',\n                        \'-ac\', \'1\', \'-ar\', \'16000\', \'-af\', \'apad\', \'-t\', str(seconds), str(target)], check=True)\n        waveform, sr = sf.read(target, dtype=\'float32\')\n        if not np.isfinite(waveform).all() or np.max(np.abs(waveform)) < 1e-5:\n            raise ValueError(\'Audio is empty or silent; upload clear speech\')\n        emb = pipe.get_audio_embedding(waveform, fps=25, device=\'cuda\', sample_rate=sr, model_type=\'avatar-v1.5\')\n        if not torch.isfinite(emb).all():\n            raise ValueError(\'Audio features contain invalid values\')\n        return emb\n\n    # Test the actual Whisper -> audio-conditioned DiT -> VAE path before readiness.\n    with torch.inference_mode():\n        emb = prepare_audio(sample_audio, 1.0, work / \'smoke-audio.wav\')\n        smoke = pipe.generate_at2v(prompt=sample_prompt, height=256, width=256,\n                                   num_frames=9, num_inference_steps=8, text_guidance_scale=1,\n                                   audio_guidance_scale=1, use_distill=True,\n                                   audio_emb=window_audio(emb, 0, 9),\n                                   generator=torch.Generator(device=\'cuda\').manual_seed(42))[0]\n        export_to_video(smoke, str(work / \'smoke.mp4\'), fps=25)\n    del emb, smoke\n    torch.cuda.empty_cache()\n    print(\'LONGCAT_AVATAR_INFERENCE_OK whisper + 8-step audio-conditioned video\', flush=True)\n\n    from longcat_avatar_jobs import JobQueue\n\n    def run_job(job_id, folder, request, update):\n        audio, image, prompt = request[\'audio\'], request[\'image\'], request[\'prompt\']\n        mode, resolution = request[\'mode\'], request[\'resolution\']\n        frames, segments, seed = request[\'frames\'], request[\'segments\'], request[\'seed\']\n        auto_duration = request[\'auto_duration\']\n        sampling = sampling_settings(request.get(\'sampling_mode\',\'distilled\'), request.get(\'steps\'),\n                                     request.get(\'text_guidance_scale\'), request.get(\'audio_guidance_scale\'))\n        started = time.time()\n        temp = work / \'jobs\' / job_id\n        temp.mkdir(parents=True,exist_ok=True)\n        with lock, torch.inference_mode():\n            try:\n                apply_sampling_mode(pipe, sampling)\n                source=audio\n                audio_seconds=None\n                if auto_duration:\n                    source=temp/\'source.wav\'\n                    subprocess.run([ffmpeg,\'-y\',\'-loglevel\',\'error\',\'-i\',str(audio),\n                                    \'-vn\',\'-ac\',\'1\',\'-ar\',\'16000\',str(source)],check=True)\n                    audio_seconds=sf.info(source).duration\n                parts,target_frames,duration=plan_segments(audio_seconds,frames,segments,auto_duration)\n                segments=len(parts)\n                update(segments=segments,duration_seconds=duration)\n                covered=parts[0]+sum(n-13 for n in parts[1:])\n                emb=prepare_audio(source,covered/25,temp/\'speech.wav\')\n                generator=torch.Generator(device=\'cuda\').manual_seed(seed)\n                checkpoint=folder/\'checkpoint.pt\'\n                next_segment=0\n                latent=ref_latent=result=None\n                if checkpoint.exists():\n                    saved=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n                    next_segment=saved[\'next_segment\']\n                    latent=saved[\'latent\'].cuda()\n                    ref_latent=saved[\'ref_latent\'].cuda()\n                    generator.set_state(saved[\'generator\'])\n                    from diffusers.utils import load_video\n                    previous=load_video(saved[\'full_video\'])\n                for segment in range(next_segment,segments):\n                    update(completed_segments=segment,message=f\'Generating segment {segment+1}/{segments}\')\n                    audio_start=sum(parts[:segment])-13*segment\n                    kwargs=dict(prompt=prompt,num_frames=parts[segment],num_inference_steps=sampling[\'steps\'],\n                                text_guidance_scale=sampling[\'text_guidance_scale\'],\n                                audio_guidance_scale=sampling[\'audio_guidance_scale\'],use_distill=sampling[\'use_distill\'],\n                                output_type=\'both\',generator=generator,\n                                audio_emb=window_audio(emb,audio_start,parts[segment]))\n                    if segment==0:\n                        if mode==\'Audio + Image + Prompt\':\n                            output,latent=pipe.generate_ai2v(image=load_image(image),resolution=resolution,**kwargs)\n                        else:\n                            height,width=(480,832) if resolution==\'480p\' else (768,1280)\n                            output,latent=pipe.generate_at2v(height=height,width=width,**kwargs)\n                        ref_latent=latent[:,:,:1].clone()\n                    else:\n                        if result is not None:\n                            previous=[Image.fromarray((f*255).clip(0,255).astype(np.uint8)) for f in result]\n                        output,latent=pipe.generate_avc(video=previous,video_latent=latent,\n                            height=previous[0].height,width=previous[0].width,\n                            num_cond_frames=13,use_kv_cache=True,offload_kv_cache=True,enhance_hf=False,\n                            ref_latent=ref_latent,ref_img_index=10,mask_frame_range=3,**kwargs)\n                    result=output[0]\n                    full=folder/f\'full-{segment:04d}.mp4\'\n                    export_to_video(result,str(full),fps=25)\n                    emitted=sum(parts[:segment])-13*max(0,segment-1) if segment else 0\n                    novel=result if segment==0 else result[13:]\n                    novel=novel[:max(0,target_frames-emitted)]\n                    export_to_video(novel,str(folder/f\'clip-{segment:04d}.mp4\'),fps=25)\n                    temporary=folder/\'checkpoint.tmp.pt\'\n                    torch.save(dict(next_segment=segment+1,latent=latent.cpu(),ref_latent=ref_latent.cpu(),\n                                    generator=generator.get_state(),full_video=str(full)),temporary)\n                    temporary.replace(checkpoint)\n                    update(completed_segments=segment+1,message=f\'Saved checkpoint for segment {segment+1}/{segments}\')\n                    for old in folder.glob(\'full-*.mp4\'):\n                        if old!=full:old.unlink()\n                listing=temp/\'clips.txt\'\n                listing.write_text(\'\'.join("file \'"+str(folder/f\'clip-{i:04d}.mp4\')+"\'\\n" for i in range(segments)))\n                destination=resolve_drive(request[\'output_dir\'],directory=True,create=True) if request.get(\'output_dir\') else output_dir\n                path=destination/(job_id+\'.mp4\')\n                subprocess.run([ffmpeg,\'-y\',\'-loglevel\',\'error\',\'-f\',\'concat\',\'-safe\',\'0\',\'-i\',str(listing),\n                                \'-i\',str(temp/\'speech.wav\'),\'-map\',\'0:v:0\',\'-map\',\'1:a:0\',\n                                \'-c:v\',\'copy\',\'-c:a\',\'aac\',\'-t\',str(duration),str(path)],check=True)\n                import av\n                with av.open(str(path)) as media:\n                    width,height=media.streams.video[0].width,media.streams.video[0].height\n                metadata=dict(job_id=job_id,model=\'LongCat-Video-Avatar-1.5\',precision=\'BF16\',mode=mode,\n                              prompt=prompt,seed=seed,**sampling,fps=25,frames=target_frames,width=width,height=height,\n                              segments=segments,auto_duration=bool(auto_duration),audio_duration_seconds=audio_seconds,\n                              duration_seconds=round(duration,2),elapsed_seconds=round(time.time()-started,1),drive_output=str(path))\n                path.with_suffix(\'.json\').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))\n                return dict(path=str(path),metadata=metadata)\n            finally:\n                pipe._clear_cache()\n                torch.cuda.empty_cache()\n                import shutil\n                shutil.rmtree(temp,ignore_errors=True)\n\n    jobs=JobQueue(output_dir.parent/\'jobs\',run_job)\n\n    def submit(audio,image,prompt,mode,resolution,frames,segments,seed,auto_duration=True,output_folder=None,\n               sampling_mode=\'distilled\',steps=None,text_guidance_scale=None,audio_guidance_scale=None):\n        if not audio or not prompt.strip():raise gr.Error(\'Upload audio and enter a prompt\')\n        if mode not in (\'Audio + Image + Prompt\',\'Audio + Prompt\') or resolution not in (\'480p\',\'720p\'):\n            raise gr.Error(\'Invalid generation mode or resolution\')\n        if mode==\'Audio + Image + Prompt\' and not image:raise gr.Error(\'Upload a reference image\')\n        frames,segments,seed=int(frames),int(segments),int(seed)\n        if not (17<=frames<=161 and frames%4==1 and 1<=segments<=20):raise gr.Error(\'Invalid manual duration parameters\')\n        if seed<0:seed=int.from_bytes(os.urandom(4),\'little\')\n        sampling=sampling_settings(sampling_mode,steps,text_guidance_scale,audio_guidance_scale)\n        request=dict(**sampling,prompt=prompt,mode=mode,resolution=resolution,frames=frames,segments=segments,\n                     seed=seed,auto_duration=bool(auto_duration))\n        if output_folder:\n            request[\'output_dir\']=str(resolve_drive(output_folder,directory=True,create=True))\n        try:job_id=jobs.submit(request,audio,image)\n        except ValueError as error:raise gr.Error(str(error))\n        return job_id,\'Job submitted; you can close this page. Job ID: \'+job_id\n\n    def output_path(state):\n        path=Path(state[\'result\'][\'path\']).resolve()\n        if not any(path.is_relative_to(root.resolve()) for root in (output_dir,DRIVE_ROOT)):\n            raise ValueError(\'Output path is outside the allowed directories\')\n        if not path.is_file():raise ValueError(\'Output video file does not exist\')\n        return path\n\n    def display_path(state):\n        path=output_path(state)\n        if path.is_relative_to(output_dir.resolve()):return str(path)\n        # Publish only this job\'s output to Gradio\'s allowlisted local cache,\n        # rather than exposing the whole mounted Drive through its file API.\n        import shutil\n        cache=work/\'published\'/path.name\n        cache.parent.mkdir(exist_ok=True)\n        if not cache.exists() or cache.stat().st_size!=path.stat().st_size:\n            temporary=cache.with_name(cache.name+\'.\'+uuid.uuid4().hex+\'.tmp\')\n            shutil.copy2(path,temporary)\n            temporary.replace(cache)\n        return str(cache)\n\n    def inspect_job(job_id):\n        if not job_id:return \'No job selected yet\',gr.skip(),gr.skip(),gr.skip()\n        try:state=jobs.get(job_id)\n        except (ValueError,FileNotFoundError):return \'Job not found\',gr.skip(),gr.skip(),gr.skip()\n        if state[\'status\']==\'completed\':\n            result=state[\'result\']\n            try:\n                published=display_path(state)\n            except (OSError,ValueError,KeyError) as error:\n                return f\'Output file unavailable: {error}\',gr.skip(),gr.skip(),result.get(\'metadata\')\n            return \'Generation complete\',published,published,result[\'metadata\']\n        progress=state.get(\'message\',state[\'status\'])\n        if state[\'status\']==\'failed\':progress=\'Job failed: \'+progress\n        return progress,gr.skip(),gr.skip(),state\n\n    with gr.Blocks(title=\'LongCat Avatar 1.5\') as demo:\n        gr.Markdown(\'# LongCat Avatar 1.5\\nUpload vocal audio and the person speaks or sings along with it. Defaults to 8 steps at 25 fps; results are saved to Google Drive automatically.\')\n        with gr.Row():\n            with gr.Column():\n                audio = gr.Audio(type=\'filepath\', sources=[\'upload\'], label=\'Vocal audio (clean dry vocals recommended)\')\n                image = gr.Image(type=\'filepath\', label=\'Reference person image\')\n                prompt = gr.Textbox(label=\'Person, scene and action description\', lines=4, value=sample_prompt)\n                mode = gr.Dropdown([\'Audio + Image + Prompt\', \'Audio + Prompt\'], value=\'Audio + Image + Prompt\', label=\'Generation mode\')\n                resolution = gr.Dropdown([\'480p\', \'720p\'], value=\'480p\', label=\'Resolution\')\n                auto_duration = gr.Checkbox(value=True, label=\'Generate for the full audio length (automatic segments)\')\n                duration_info = gr.Markdown(\'Upload audio to see its full length and the estimated number of segments.\')\n                with gr.Accordion(\'Manual duration and random seed\', open=False):\n                    frames = gr.Slider(17, 161, value=93, step=4, label=\'Manual mode: frames per segment\', interactive=False)\n                    segments = gr.Slider(1, 20, value=1, step=1, label=\'Manual mode: number of segments\', interactive=False)\n                    seed = gr.Number(value=42, precision=0, label=\'Random seed (-1 for random)\')\n                with gr.Accordion(\'Lip sync and sampling settings\', open=False):\n                    sampling_mode = gr.Dropdown(choices=[(\'Fast - 8-step distilled\',\'distilled\'),(\'Standard - non-distilled, adjustable CFG\',\'standard\')],\n                                               value=\'distilled\', label=\'Sampling mode\')\n                    steps = gr.Slider(8,100,value=8,step=1,label=\'Inference steps\',interactive=False)\n                    text_cfg = gr.Slider(1,10,value=1,step=0.1,label=\'Text CFG\',interactive=False)\n                    audio_cfg = gr.Slider(1,10,value=1,step=0.1,label=\'Audio CFG\',interactive=False)\n                    gr.Markdown(\'Standard mode is slower. Audio CFG controls audio guidance; lowering it may hurt sync. It does not directly control mouth amplitude.\')\n                button = gr.Button(\'Submit generation job\', variant=\'primary\')\n            with gr.Column():\n                job_id = gr.Textbox(label=\'Job ID (paste it here after reopening the page)\')\n                history = gr.Dropdown(label=\'Job history\', choices=jobs.recent())\n                status = gr.Markdown(\'After submission the server generates in the background; closing the page does not affect the job.\')\n                video = gr.Video(label=\'Result\')\n                download = gr.File(label=\'Download video with audio\')\n                metadata = gr.JSON(label=\'Generation info\')\n        gr.Examples([[str(sample_audio), str(sample_image), sample_prompt]], [audio, image, prompt], label=\'Official example\')\n        def describe_duration(path, automatic):\n            if not automatic:\n                return \'Manual mode trims the audio to the configured frames and segments.\'\n            if not path:\n                return \'Upload audio to see its full length and the estimated number of segments.\'\n            try:\n                seconds = sf.info(path).duration\n                parts, _, _ = plan_segments(seconds)\n                return f\'Audio is about **{seconds:.2f} s**; **{len(parts)} segments** will be generated automatically, covering the full audio.\'\n            except Exception as error:\n                return str(error)\n        audio.change(describe_duration, [audio,auto_duration], duration_info)\n        auto_duration.change(describe_duration, [audio,auto_duration], duration_info)\n        auto_duration.change(lambda enabled: [gr.update(interactive=not enabled)]*2,\n                             auto_duration, [frames,segments])\n        def change_sampling(value):\n            settings=sampling_settings(value)\n            return [gr.update(value=settings[key],interactive=value==\'standard\')\n                    for key in (\'steps\',\'text_guidance_scale\',\'audio_guidance_scale\')]\n        def submit_ui(audio,image,prompt,mode,resolution,frames,segments,seed,auto_duration=True,\n                      sampling_mode=\'distilled\',steps=None,text_cfg=None,audio_cfg=None):\n            return submit(audio,image,prompt,mode,resolution,frames,segments,seed,auto_duration,\n                          None,sampling_mode,steps,text_cfg,audio_cfg)\n        button.click(submit_ui,\n                     [audio,image,prompt,mode,resolution,frames,segments,seed,auto_duration,sampling_mode,steps,text_cfg,audio_cfg],\n                     [job_id,status],api_name=\'submit\',concurrency_limit=4)\n        timer=gr.Timer(3)\n        timer.tick(inspect_job,job_id,[status,video,download,metadata],api_name=False,queue=False)\n        history.change(lambda value:value,history,job_id)\n        demo.load(lambda:gr.update(choices=jobs.recent(),value=jobs.recent()[0] if jobs.recent() else None),\n                  outputs=history)\n        sampling_mode.change(change_sampling,sampling_mode,[steps,text_cfg,audio_cfg])\n    api = FastAPI()\n\n    @api.get(\'/health\')\n    def health():\n        return dict(status=\'ok\', model=\'LongCat-Video-Avatar-1.5\', precision=\'BF16\',\n                    gpu=torch.cuda.get_device_name(0), inference_verified=True,\n                    active_job=jobs.active,draining=jobs.draining,sampling_modes=[\'distilled\',\'standard\'])\n\n    @api.post(\'/jobs\',status_code=202)\n    def create_job(body: dict):\n        try:\n            audio=resolve_input(body.get(\'audio\'))\n            image=resolve_input(body[\'image\']) if body.get(\'image\') else None\n            automatic=body.get(\'auto_duration\',True)\n            if not isinstance(automatic,bool):raise ValueError(\'auto_duration must be a boolean\')\n            job_id,_=submit(str(audio),str(image) if image else None,str(body.get(\'prompt\',\'\')),\n                \'Audio + Image + Prompt\' if image else \'Audio + Prompt\',body.get(\'resolution\',\'480p\'),\n                body.get(\'frames\',93),body.get(\'segments\',1),body.get(\'seed\',42),automatic,body.get(\'output_dir\'),\n                body.get(\'sampling_mode\',\'distilled\'),body.get(\'steps\'),\n                body.get(\'text_guidance_scale\'),body.get(\'audio_guidance_scale\'))\n            return {\'job_id\':job_id}\n        except (ValueError,OSError,gr.Error) as error:raise HTTPException(400,str(error))\n\n    @api.get(\'/drive/files\')\n    def drive_files(path: str=\'drive://\'):\n        try:return {\'files\':list_drive(path)}\n        except (ValueError,OSError) as error:raise HTTPException(400,str(error))\n\n    @api.get(\'/jobs/{job_id}\')\n    def get_job(job_id: str):\n        try:return jobs.get(job_id)\n        except (ValueError,FileNotFoundError):raise HTTPException(404,\'Job not found\')\n\n    @api.get(\'/jobs/{job_id}/video\')\n    def job_video(job_id: str):\n        state=get_job(job_id)\n        if state[\'status\']!=\'completed\':raise HTTPException(409,\'Job not finished yet\')\n        try:path=output_path(state)\n        except ValueError as error:raise HTTPException(404,str(error))\n        return FileResponse(path,media_type=\'video/mp4\',filename=path.name)\n\n    demo.queue(max_size=8, default_concurrency_limit=1)\n    host = os.environ.get(\'CT_PUBLIC_HOSTNAME\',\'\')\n    api = gr.mount_gradio_app(api, demo, path=\'/\', allowed_paths=[str(output_dir), str(work/\'published\'), str(sample_audio), str(sample_image)],\n                             root_path=f\'https://{host}\' if host else None)\n    # SIGTERM drains accepted jobs before shutting down the HTTP server. A hard\n    # stop can resume from the most recent per-segment checkpoint on startup.\n    import signal\n    from contextlib import contextmanager\n    class JobServer(uvicorn.Server):\n        @contextmanager\n        def capture_signals(self):\n            def shutdown(signum,frame):\n                jobs.drain()\n                if jobs.shutdown_ready.is_set():self.should_exit=True\n            previous={sig:signal.signal(sig,shutdown) for sig in (signal.SIGINT,signal.SIGTERM)}\n            try:yield\n            finally:\n                for sig,handler in previous.items():signal.signal(sig,handler)\n    server=JobServer(uvicorn.Config(api,host=\'127.0.0.1\',port=int(os.environ.get(\'CT_SERVICE_PORT\',\'7860\'))))\n    def await_shutdown():\n        jobs.shutdown_ready.wait()\n        server.should_exit=True\n    jobs.start()\n    threading.Thread(target=await_shutdown,daemon=True).start()\n    try:server.run()\n    finally:\n        jobs.stop.set()\n        jobs.thread.join()\n\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")
(LC_WORK / 'longcat_avatar_jobs.py').write_text('"""Persistent, browser-independent serial jobs for LongCat Avatar."""\nimport json\nimport shutil\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name+\'.\'+uuid.uuid4().hex+\'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2))\n    temporary.replace(path)\n\n\nclass JobQueue:\n    def __init__(self, root, runner):\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.runner = runner\n        self.lock = threading.RLock()\n        self.active = None\n        self.draining = False\n        self.shutdown_ready = threading.Event()\n        self.stop = threading.Event()\n        for path in self.root.glob(\'*/status.json\'):\n            try:\n                value = json.loads(path.read_text())\n                if value.get(\'status\')==\'running\':\n                    value.update(status=\'queued\', message=\'Resuming interrupted job from the latest checkpoint\')\n                    write_json(path,value)\n            except (OSError, ValueError, KeyError, TypeError) as error:\n                print(f\'Warning: skipping unreadable job status {path}: {error}\', flush=True)\n\n    def submit(self, request, audio, image=None):\n        with self.lock:\n            if self.draining:\n                raise ValueError(\'The service is updating; submit new jobs later\')\n            job_id = time.strftime(\'%Y%m%d-%H%M%S\')+\'-\'+uuid.uuid4().hex[:8]\n            folder = self.root/job_id\n            folder.mkdir()\n            audio_path = folder/(\'input-audio\'+Path(audio).suffix)\n            shutil.copy2(audio,audio_path)\n            request = {**request, \'audio\':str(audio_path), \'image\':None}\n            if image:\n                image_path=folder/(\'input-image\'+Path(image).suffix)\n                shutil.copy2(image,image_path)\n                request[\'image\']=str(image_path)\n            write_json(folder/\'request.json\',request)\n            write_json(folder/\'status.json\',dict(id=job_id,status=\'queued\',message=\'Queued\',created=time.time(),attempts=0))\n            return job_id\n\n    def get(self, job_id):\n        if not job_id or Path(job_id).name!=job_id:\n            raise ValueError(\'Invalid job ID\')\n        return json.loads((self.root/job_id/\'status.json\').read_text())\n\n    def recent(self):\n        return sorted([p.name for p in self.root.iterdir() if (p/\'status.json\').exists()],reverse=True)\n\n    def update(self, job_id, **values):\n        with self.lock:\n            state=self.get(job_id)\n            state.update(values,updated=time.time())\n            write_json(self.root/job_id/\'status.json\',state)\n            return state\n\n    def drain(self):\n        with self.lock:\n            self.draining=True\n\n    def start(self):\n        self.thread=threading.Thread(target=self._run,name=\'avatar-job-worker\',daemon=False)\n        self.thread.start()\n\n    def _run(self):\n        while not self.stop.is_set():\n            with self.lock:\n                pending=[]\n                for job in reversed(self.recent()):\n                    try:\n                        if self.get(job).get(\'status\')==\'queued\':pending.append(job)\n                    except (FileNotFoundError, OSError, ValueError, KeyError, TypeError):\n                        # Completed validation/history folders may be archived\n                        # or unreadable while scanning the shared Drive.\n                        continue\n                if not pending:\n                    if self.draining:\n                        self.shutdown_ready.set()\n                        return\n                    job_id=None\n                else:\n                    job_id=pending[0]\n                    state=self.get(job_id)\n                    self.active=job_id\n                    self.update(job_id,status=\'running\',attempts=state[\'attempts\']+1,message=\'Generating\')\n            if not job_id:\n                self.stop.wait(.5)\n                continue\n            try:\n                folder=self.root/job_id\n                request=json.loads((folder/\'request.json\').read_text())\n                result=self.runner(job_id,folder,request,lambda **kw:self.update(job_id,**kw))\n                self.update(job_id,status=\'completed\',message=\'Generation complete\',result=result)\n            except Exception as error:\n                self.update(job_id,status=\'failed\',message=str(error))\n                import traceback\n                traceback.print_exc()\n            finally:\n                with self.lock:self.active=None\n', encoding="utf-8")
(LC_WORK / 'longcat_avatar_drive.py').write_text('"""Resolve mounted Google Drive paths and validated Gradio uploads."""\nimport os\nfrom pathlib import Path\nfrom urllib.parse import unquote\n\nDRIVE_ROOT=Path(\'/content/drive/MyDrive\')\nUPLOAD_ROOT=Path(os.environ.get(\'GRADIO_TEMP_DIR\',\'/tmp/gradio\'))\n\n\ndef resolve_drive(spec, *, directory=False, create=False, root=DRIVE_ROOT):\n    root=Path(root).resolve()\n    if not root.is_dir():raise ValueError(\'Google Drive is not mounted\')\n    text=str(spec)\n    for prefix in (\'drive://\',\'gdrive://\'):\n        if text.startswith(prefix):text=unquote(text[len(prefix):]);break\n    if text.startswith(\'MyDrive/\'):text=text[len(\'MyDrive/\'):]\n    candidate=Path(text)\n    if not candidate.is_absolute():candidate=root/candidate\n    candidate=candidate.resolve()\n    if not candidate.is_relative_to(root):raise ValueError(\'The path must be inside MyDrive on Google Drive\')\n    if directory:\n        if create:candidate.mkdir(parents=True,exist_ok=True)\n        if not candidate.is_dir():raise ValueError(f\'Drive directory does not exist: {spec}\')\n    elif not candidate.is_file():raise ValueError(f\'Drive file does not exist: {spec}\')\n    return candidate\n\n\ndef resolve_input(reference, *, drive_root=DRIVE_ROOT, upload_root=UPLOAD_ROOT):\n    if not isinstance(reference,dict) or len(reference)!=1:\n        raise ValueError(\'The input must specify a drive or upload path\')\n    if \'drive\' in reference:return resolve_drive(reference[\'drive\'],root=drive_root)\n    if \'upload\' in reference:\n        path=Path(reference[\'upload\']).resolve()\n        if not path.is_relative_to(Path(upload_root).resolve()) or not path.is_file():\n            raise ValueError(\'Uploaded files must be inside the service upload cache\')\n        return path\n    raise ValueError(\'Unsupported input path type\')\n\n\ndef list_drive(spec=\'drive://\', *, root=DRIVE_ROOT):\n    folder=resolve_drive(spec,directory=True,root=root)\n    base=Path(root).resolve()\n    return [{\'name\':p.name,\'type\':\'directory\' if p.is_dir() else \'file\',\n             \'path\':\'drive://\'+p.relative_to(base).as_posix(),\n             \'size\':p.stat().st_size if p.is_file() else None}\n            for p in sorted(folder.iterdir(),key=lambda p:(not p.is_dir(),p.name))\n            if p.resolve().is_relative_to(base)]\n', encoding="utf-8")
(LC_WORK / 'avatar_sampling.py').write_text('"""Validated sampling controls shared by the Avatar CLI and GPU worker."""\nimport math\n\n\ndef sampling_settings(mode=\'distilled\', steps=None, text_cfg=None, audio_cfg=None):\n    if mode not in (\'distilled\', \'standard\'):\n        raise ValueError(\'sampling_mode must be distilled or standard\')\n    defaults = (8, 1.0, 1.0) if mode == \'distilled\' else (50, 4.0, 3.0)\n    steps = defaults[0] if steps is None else steps\n    text_cfg = defaults[1] if text_cfg is None else text_cfg\n    audio_cfg = defaults[2] if audio_cfg is None else audio_cfg\n    if isinstance(steps, bool) or not isinstance(steps, (int, float)) or not math.isfinite(steps) or int(steps) != steps or not 8 <= steps <= 100:\n        raise ValueError(\'steps must be an integer from 8 to 100\')\n    for name, value in ((\'text_guidance_scale\', text_cfg), (\'audio_guidance_scale\', audio_cfg)):\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 1 <= value <= 10:\n            raise ValueError(name + \' must be a finite number from 1 to 10\')\n    if mode == \'distilled\' and (steps, text_cfg, audio_cfg) != defaults:\n        raise ValueError(\'8-step distilled mode fixes steps=8 and text/audio CFG=1; use standard mode to adjust CFG\')\n    return dict(sampling_mode=mode, steps=int(steps), text_guidance_scale=float(text_cfg),\n                audio_guidance_scale=float(audio_cfg), use_distill=mode == \'distilled\')\n\n\ndef apply_sampling_mode(pipe, settings):\n    """Called under the worker lock, before generation or checkpoint resume."""\n    if settings[\'use_distill\']:\n        pipe.dit.enable_loras([\'dmd\'])\n    else:\n        pipe.dit.disable_all_loras()\n\n\ndef plan_segments(audio_seconds, frames=93, segments=1, auto_duration=True):\n    if not auto_duration:\n        total = frames + (segments - 1) * (frames - 13)\n        return [frames] * segments, total, total / 25\n    if not math.isfinite(audio_seconds) or audio_seconds <= 0:\n        raise ValueError(\'Invalid audio duration\')\n    target = max(1, math.ceil(round(audio_seconds * 25, 8)))\n    # Each model call needs 4n+1 frames; subsequent calls include 13\n    # overlapping conditioning frames. Shorten the final call and trim padding.\n    parts = [max(17, 1 + 4 * math.ceil((min(target, 93) - 1) / 4))]\n    covered = parts[0]\n    while covered < target:\n        count = min(93, max(17, 13 + 4 * math.ceil((target - covered) / 4)))\n        parts.append(count)\n        covered += count - 13\n        if len(parts) > 100:\n            raise ValueError(\'Audio is too long; split it into clips of at most 5 minutes before generating\')\n    return parts, target, audio_seconds\n', encoding="utf-8")
LC_APP = LC_WORK / 'longcat_avatar_app.py'

def start_service():
    return ct_spawn("service", [LC_PYTHON, str(LC_APP)], cwd=str(LC_REPO), env={
        "PYTHONPATH": str(LC_REPO), "PYTHONUNBUFFERED": "1",
        "CT_LONGCAT_MODEL_DIR": str(LC_MODEL), "CT_LONGCAT_WORK_DIR": str(LC_WORK),
        "CT_LONGCAT_OUTPUT_DIR": str(LC_OUTPUT),
        "CT_SERVICE_PORT": str(SERVICE_PORT), "CT_LONGCAT_REPO_DIR": str(LC_REPO),
        "TRITON_CACHE_DIR": str(LC_WORK / "triton-cache"),
        "TORCHINDUCTOR_CACHE_DIR": str(LC_WORK / "inductor-cache"),
        "GRADIO_ANALYTICS_ENABLED": "False",
    })

PROCS["service"] = (start_service(), start_service)
# Wait for model + real inference before the shared authentication/tunnel cells.
deadline = time.time() + 1800
while time.time() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        raise RuntimeError("LongCat service exited:\n" + ct_tail("service", 60))
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=3) as response:
            if json.load(response).get("inference_verified"):
                break
    except (OSError, ValueError):
        pass
    print("LongCat loading / warming up:\n" + ct_tail("service", 3), flush=True)
    time.sleep(20)
else:
    raise TimeoutError("LongCat inference readiness timed out:\n" + ct_tail("service", 60))
for local_cache in LC_CACHE_PAIRS:
    try:
        save_cache(local_cache, LC_CACHE_ARCHIVES, LC_CACHE_KEY)
    except (OSError, ValueError) as err:
        print(f"Cache archive save warning ({local_cache.name}): {err}", flush=True)
print("LongCat inference verified; compiled kernels archived to Drive", flush=True)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = False   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            header_up X-Forwarded-Proto https
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- KEEP THIS CELL RUNNING
# The Colab session only lives while the `colab exec` connection is open and this cell blocks.
# The watchdog restarts any process from PROCS that died; a heartbeat line is printed every 10 min.
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=3600)
if AUTH == "caddy" and not CREDS_FROM_ENV:   # started from the Colab UI: show generated credentials
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

last_beat = time.time()
while True:
    time.sleep(30)
    for name, (proc, restart) in list(PROCS.items()):
        if proc.poll() is not None:
            print(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 15)}", flush=True)
            ct_stop_proc(proc)
            PROCS[name] = (restart(), restart)
    if time.time() - last_beat > 600:
        last_beat = time.time()
        print(f"[heartbeat] {time.strftime('%H:%M:%S')} " +
              " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items()), flush=True)
